《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 누적분포함수

도수분포표와 PMF는 분포를 나타내는 가장 익숙한 방법이지만, 이 장에서 보겠듯이 한계가 있습니다. 다른 방법으로 누적분포함수(CDF)가 있습니다. 백분위수를 계산할 때 유용하고, 특히 분포를 비교할 때 좋습니다.

이 장에서는 백분위수에 바탕을 둔 통계량으로 분포의 위치, 퍼진 정도, 치우친 정도도 수치로 나타내 보겠습니다.

[이 노트북의 원문을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/v3/nb/chap04.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 백분위수와 백분위 순위

표준화 시험을 봤다면 원점수와 **백분위 순위**(percentile rank)를 함께 받아 본 적이 있을 것입니다. 여기서 백분위 순위는 자신과 같거나 낮은 점수를 받은 사람의 비율을 백분율로 나타낸 값입니다. “90백분위에 속한다”면, 응시자의 90%와 같거나 그보다 높은 점수를 받았다는 뜻입니다.

백분위수와 백분위 순위를 이해하기 위해 달리기 속도를 예로 들어 보겠습니다. 몇 년 전 저는 매사추세츠에서 열리는 10km 도로 달리기 대회인 James Joyce Ramble에 참가했습니다. 대회가 끝난 뒤, 제 기록이 다른 참가자와 비교해 어느 정도인지 알아보려고 결과를 내려받았습니다.

자료를 내려받는 방법은 이 장의 노트북에 있습니다.

In [4]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/relay.py")
download(
    "https://github.com/AllenDowney/ThinkStats/raw/v3/data/Apr25_27thAn_set1.shtml"
)

`relay.py` 모듈에는 결과를 읽어 Pandas `DataFrame`으로 반환하는 함수가 있습니다.

In [5]:
from relay import read_results

results = read_results()
results.head()

`results`에는 완주자 1,633명 각각에 해당하는 행이 있습니다. 기록을 비교할 때 사용할 열은 `MPH`입니다. 각 주자의 평균 속도가 시속 마일 단위로 들어 있습니다. 이 열을 선택하고 `values`로 속도들을 NumPy 배열로 꺼내겠습니다.

In [6]:
speeds = results["MPH"].values

제 완주 기록은 42분 44초였으므로 다음과 같이 제 행을 찾을 수 있습니다.

In [7]:
my_result = results.query("Nettime == '42:44'")
my_result

제 행의 인덱스는 96이므로, 속도를 다음과 같이 꺼낼 수 있습니다.

In [8]:
my_speed = speeds[96]

`sum`으로 저와 같거나 느린 속도로 달린 주자의 수를 셀 수 있습니다.

In [9]:
(speeds <= my_speed).sum()

`mean`을 사용하면 그 비율을 구할 수 있고, 100을 곱하면 백분율이 됩니다.

In [10]:
(speeds <= my_speed).mean() * 100

결과는 전체 참가자 중 제 백분위 순위인 약 94%입니다.

일반적으로 다음 함수는 값의 시퀀스에서 특정 값의 백분위 순위를 구합니다.

In [11]:
def percentile_rank(x, seq):
    """Percentile rank of x.

    x: value
    seq: sequence of values

    returns: percentile rank 0-100
    """
    return (seq <= x).mean() * 100

`results`의 `Division` 열에는 성별과 연령대로 나눈 참가 부문이 들어 있습니다. 예를 들어 저는 40~49세 남성인 `M4049` 부문에 속했습니다. `query` 메서드로 같은 부문의 행을 선택하고 속도들을 꺼낼 수 있습니다.

In [12]:
my_division = results.query("Division == 'M4049'")
my_division_speeds = my_division["MPH"].values

이제 `percentile_rank`로 제 부문 안에서의 백분위 순위를 구할 수 있습니다.

In [13]:
percentile_rank(my_speed, my_division_speeds)

반대 방향으로도 구할 수 있습니다. 백분위 순위가 주어졌을 때, 다음 함수는 시퀀스에서 그에 대응하는 값을 찾습니다.

> 번역자 주: 아래 함수는 이 대회 자료가 속도 내림차순으로 정렬되어 있다는 점을 이용합니다. 일반적인 자료에는 정렬과 경계값 처리가 필요합니다. 뒤에서 다루는 `Cdf.inverse`를 사용하면 이런 순서에 의존하지 않고 조회할 수 있습니다.

In [14]:
def percentile(p, seq):
    n = len(seq)
    i = (1 - p / 100) * (n + 1)
    return seq[round(i)]

`n`은 시퀀스의 원소 수이고, `i`는 해당 백분위 순위에 있는 원소의 인덱스입니다. 백분위 순위로 조회했을 때 나오는 값을 **백분위수**(percentile)라고 합니다.

In [15]:
percentile(90, my_division_speeds)

제 부문에서 90백분위수는 시속 약 8.6마일이었습니다.

그 대회에서 달린 뒤 몇 년이 지나, 이제 저는 `M5059` 부문에 속합니다. 새 부문에서 같은 백분위 순위를 얻으려면 얼마나 빨리 달려야 할까요? `M4049` 부문에서의 백분위 순위인 약 90.2%를 `M5059` 부문의 속도로 바꾸면 답을 구할 수 있습니다.

In [16]:
next_division = results.query("Division == 'M5059'")
next_division_speeds = next_division["MPH"].values

percentile(90.2, next_division_speeds)

`M5059` 부문에서 저와 백분위 순위가 같은 사람은 시속 8마일을 조금 넘는 속도로 달렸습니다. `query`로 그 사람을 찾을 수 있습니다.

In [17]:
next_division.query("MPH > 8.01").tail(1)

이 사람은 46분 25초에 완주했고, 같은 부문 참가자 171명 중 18위였습니다.

백분위 순위와 백분위수를 살펴보았으니, 이제 누적분포함수로 넘어가겠습니다.

## CDF

**누적분포함수**(cumulative distribution function, CDF)는 도수분포표나 PMF와 마찬가지로 값들의 분포를 설명하는 방법입니다. 값 `x`가 주어지면, `x` 이하인 값의 비율을 계산합니다. 짧은 시퀀스를 예로 들어 보겠습니다.

In [18]:
t = [1, 2, 2, 3, 5]

CDF를 구하는 한 가지 방법은 PMF에서 출발하는 것입니다. 다음 `Pmf`는 `t`에 들어 있는 값들의 분포를 나타냅니다.

In [19]:
from empiricaldist import Pmf

pmf = Pmf.from_seq(t)
pmf

앞 장에서 보았듯이 대괄호 연산자로 `Pmf`에서 값을 조회할 수 있습니다.

In [20]:
pmf[2]

결과는 주어진 값과 같은 값들이 차지하는 비율입니다. 이 예에서는 다섯 값 중 두 개가 `2`이므로 결과는 0.4입니다. 시퀀스에서 무작위로 고른 값이 `2`일 확률이라고 생각해도 됩니다.

`Pmf`의 `make_cdf` 메서드는 확률의 누적합을 계산합니다.

In [21]:
cdf = pmf.make_cdf()
cdf

결과는 Pandas `Series`의 한 종류인 `Cdf` 객체입니다. 대괄호 연산자로 값을 조회할 수 있습니다.

In [22]:
cdf[2]

결과는 주어진 값 이하인 값들이 차지하는 비율입니다. 이 예에서는 다섯 값 중 세 개가 `2` 이하이므로 결과는 0.6입니다. 시퀀스에서 무작위로 고른 값이 `2` 이하일 확률이라고 생각해도 됩니다.

괄호로 `Cdf` 객체를 함수처럼 호출할 수도 있습니다.

In [23]:
cdf(3)

누적분포함수는 시퀀스에 실제로 나타난 값뿐 아니라 모든 수에 대해 정의됩니다.

In [24]:
cdf(4)

`step` 메서드는 `Cdf`를 계단 함수로 그립니다.

In [25]:
cdf.step()
decorate(xlabel="x", ylabel="CDF")

두 번째 예로 앞 절의 달리기 속도 분포를 나타내는 `Cdf`를 만들겠습니다. `Cdf` 클래스의 `from_seq` 함수로 시퀀스에서 `Cdf` 객체를 만들 수 있습니다.

In [26]:
from empiricaldist import Cdf

cdf_speeds = Cdf.from_seq(speeds)

그림으로 나타내면 다음과 같습니다. 세로선은 제 속도에 그었습니다.

In [27]:
cdf_speeds.step()
plt.axvline(my_speed, ls=":", color="gray")
decorate(xlabel="Speed (mph)", ylabel="CDF")

제 속도를 조회하면 저와 같거나 느린 주자의 비율이 나옵니다. 100을 곱하면 제 백분위 순위가 됩니다.

In [28]:
cdf_speeds(my_speed) * 100

이렇게 보면 `Cdf`는 값을 받아 백분위 순위와 비슷한 것을 계산합니다. 다만 0~100의 백분율 대신 0~1의 비율을 반환합니다.

`inverse` 메서드는 누적분포함수를 거꾸로 계산합니다. 0~1 사이의 비율을 받아 그에 대응하는 값을 찾습니다.

예를 들어 전체 주자의 50%와 같거나 더 빠르게 달렸다는 사람의 속도는 다음과 같이 찾을 수 있습니다.

In [29]:
cdf_speeds.inverse(0.5)

비율이 주어졌을 때 역 CDF로 찾은 값을 **분위수**(quantile)라고 합니다. 그래서 역 CDF를 분위수 함수라고도 합니다.

반대로 분위수를 CDF에 넣어 구한 비율에는, 이상하게도 딱 정해진 이름이 없습니다. 백분위수와 백분위 순위에 맞추어 “분위 순위”라고 부를 수도 있겠지만, 제가 알기로 그렇게 부르는 사람은 없습니다. 대개는 그냥 “누적확률”이라고 합니다.

## CDF 비교하기

CDF는 특히 분포를 비교할 때 유용합니다. 예로 첫아이와 다른 아이의 출생 체중 분포를 비교하겠습니다. NSFG 자료를 다시 불러와 생존 출산 전체, 첫아이, 나머지를 담은 세 `DataFrame`으로 나눕니다.

다음 셀들은 데이터 파일을 내려받고, 자료를 읽는 데 필요한 `statadict`를 설치합니다.

In [30]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [31]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [32]:
from nsfg import get_nsfg_groups

live, firsts, others = get_nsfg_groups()

`firsts`와 `others`에서 파운드 단위의 총 출생 체중을 선택합니다. `dropna`로 `nan`인 값을 제외합니다.

In [33]:
first_weights = firsts["totalwgt_lb"].dropna()
first_weights.mean()

In [34]:
other_weights = others["totalwgt_lb"].dropna()
other_weights.mean()

첫아이의 체중이 평균적으로 조금 가벼워 보입니다. 하지만 이런 차이가 생기는 방식은 여러 가지입니다. 유난히 가벼운 첫아이가 소수 있거나, 유난히 무거운 다른 아이가 소수 있을 수 있습니다. 이런 경우에는 분포의 모양이 달라질 것입니다. 또는 모양은 같고 위치만 다를 수도 있습니다.

분포를 비교하려고 PMF를 그려 볼 수 있습니다.

In [35]:
from empiricaldist import Pmf

first_pmf = Pmf.from_seq(first_weights, name="first")
other_pmf = Pmf.from_seq(other_weights, name="other")

하지만 다음 그림에서 보듯 그리 잘 되지는 않습니다.

In [36]:
from thinkstats import two_bar_plots

two_bar_plots(first_pmf, other_pmf, width=0.06)
decorate(xlabel="Weight (pounds)", ylabel="PMF")

분포를 최대한 분명하게 보이려고 막대 너비와 투명도를 조절했지만, 두 분포를 비교하기 어렵습니다. 봉우리와 골이 많고 차이처럼 보이는 부분도 있지만, 무엇이 의미 있는 특징인지 판단하기 어렵습니다. 전체적인 패턴도 잘 보이지 않습니다. 예를 들어 어느 분포의 평균이 더 높은지 눈으로는 분명하지 않습니다.

값의 범위를 서로 겹치지 않는 구간으로 나누고, 각 구간에 들어가는 값의 개수를 세는 **구간화**(binning)를 하면 문제를 줄일 수 있습니다. 유용한 방법이지만 적절한 구간 너비를 정하기가 까다롭습니다. 잡음을 매끈하게 만들 만큼 구간을 넓히면, 유용한 정보까지 사라질 수 있습니다.

좋은 대안은 CDF를 그리는 것입니다.

In [37]:
first_cdf = first_pmf.make_cdf()
other_cdf = other_pmf.make_cdf()

결과는 다음과 같습니다.

In [38]:
first_cdf.plot(ls="--")
other_cdf.plot(alpha=0.5)
decorate(xlabel="Weight (pounds)", ylabel="CDF")

이 그림에서는 분포의 모양과 두 분포 사이의 차이가 훨씬 분명합니다. 첫아이의 곡선은 나머지 아이의 곡선보다 일관되게 왼쪽에 있습니다. 분포 전반에서 첫아이가 조금 더 가볍고, 중간보다 높은 쪽에서 차이가 더 크다는 뜻입니다.

## 백분위수에 바탕을 둔 통계량

[3장](/posts/statistics/think-stats-03#pmf-요약하기)에서는 분포의 중심을 나타내는 산술평균과, 퍼진 정도를 나타내는 표준편차를 구했습니다. 앞선 연습문제에서는 분포가 왼쪽이나 오른쪽으로 치우친 정도인 왜도도 구했습니다. 이 통계량들의 단점은 이상치에 민감하다는 것입니다. 극단적인 값 하나가 평균, 표준편차, 왜도에 큰 영향을 줄 수 있습니다.

대안은 분포의 백분위수에 바탕을 둔 통계량입니다. 이런 통계량은 대체로 더 **강건한**(robust) 통계량입니다. 이상치의 영향을 덜 받는다는 뜻입니다. 이를 보이려고 이번에는 아무 정제도 하지 않고 NSFG 자료를 다시 읽겠습니다.

In [39]:
from nsfg import read_stata

dct_file = "2002FemPreg.dct"
dat_file = "2002FemPreg.dat.gz"

preg = read_stata(dct_file, dat_file)

출생 체중은 파운드 부분과 온스 부분, 두 열에 나뉘어 기록되어 있었습니다.

In [40]:
birthwgt_lb = preg["birthwgt_lb"]
birthwgt_oz = preg["birthwgt_oz"]

`birthwgt_oz`로 `Hist` 객체를 만들면 결측을 뜻하는 특수한 값 97, 98, 99가 들어 있음을 확인할 수 있습니다.

In [41]:
from empiricaldist import Hist

Hist.from_seq(birthwgt_oz).tail(5)

`birthwgt_lb`에도 같은 특수 값이 있습니다. 분명 오류인 51도 들어 있습니다.

In [42]:
Hist.from_seq(birthwgt_lb).tail(5)

두 상황을 생각해 보겠습니다. 첫 번째는 결측이나 잘못된 값을 `nan`으로 바꿔 정제하고, 총체중을 파운드 단위로 계산하는 상황입니다. `birthwgt_oz_clean`을 16으로 나누면 파운드의 소수 부분으로 바뀝니다.

In [43]:
birthwgt_lb_clean = birthwgt_lb.replace([51, 97, 98, 99], np.nan)
birthwgt_oz_clean = birthwgt_oz.replace([97, 98, 99], np.nan)

total_weight_clean = birthwgt_lb_clean + birthwgt_oz_clean / 16

두 번째는 자료를 정제하지 않은 채 잘못된 값까지 포함해 총체중을 계산하는 상황입니다.

In [44]:
total_weight_bogus = birthwgt_lb + birthwgt_oz / 16

정제하지 않은 자료에만 들어 있는 잘못된 값은 49개로, 자료의 약 0.5%입니다.

In [45]:
count1, count2 = total_weight_bogus.count(), total_weight_clean.count()
diff = count1 - count2

diff, diff / count2 * 100

이제 두 상황에서 자료의 평균을 구해 보겠습니다.

In [46]:
mean1, mean2 = total_weight_bogus.mean(), total_weight_clean.mean()
mean1, mean2

잘못된 값이 평균에 미치는 영향은 그리 크지 않습니다. 정제한 자료의 평균을 올바른 값으로 보면, 정제하지 않은 자료의 평균은 1% 미만으로 어긋납니다.

In [47]:
(mean1 - mean2) / mean2 * 100

이 정도 오류는 알아차리지 못할 수도 있습니다. 하지만 표준편차에서는 어떤 일이 생기는지 보겠습니다.

In [48]:
std1, std2 = total_weight_bogus.std(), total_weight_clean.std()
std1, std2

In [49]:
(std1 - std2) / std2 * 100

정제하지 않은 자료의 표준편차는 거의 50%나 어긋나므로 더 눈에 띕니다. 마지막으로 두 자료의 왜도를 보겠습니다.

In [50]:
def skewness(seq):
    """Compute the skewness of a sequence

    seq: sequence of numbers

    returns: float skewness
    """
    deviations = seq - seq.mean()
    return np.mean(deviations**3) / seq.std(ddof=0) ** 3

In [51]:
skew1, skew2 = skewness(total_weight_bogus), skewness(total_weight_clean)
skew1, skew2

In [52]:
# how much is skew1 off by?
(skew1 - skew2) / skew2

정제하지 않은 자료의 왜도는 크기가 거의 40배나 어긋나고, 부호까지 반대입니다! 이상치가 섞이면 왜도가 큰 양수가 되어 분포가 오른쪽으로 강하게 치우칩니다. 올바른 자료의 왜도는 작은 음수로, 왼쪽으로 조금 치우쳐 있습니다.

이 결과에서 소수의 이상치는 평균에는 어느 정도, 표준편차에는 크게, 왜도에는 매우 크게 영향을 줍니다.

백분위수에 바탕을 둔 통계량을 대안으로 쓸 수 있습니다.

- 중앙값은 50백분위수이며, 평균처럼 분포의 중심을 나타냅니다.
- 사분위범위는 75백분위수와 25백분위수의 차이이며, 표준편차처럼 분포의 퍼진 정도를 나타냅니다.
- 사분위 왜도는 25·50·75백분위수를 이용해 치우친 정도를 수치로 나타냅니다.

`Cdf` 객체는 이런 통계량을 효율적으로 계산하게 해 줍니다. 정제하지 않은 자료와 정제한 자료에서 각각 `Cdf` 객체를 만들겠습니다.

In [53]:
cdf_total_weight_bogus = Cdf.from_seq(total_weight_bogus)
cdf_total_weight_clean = Cdf.from_seq(total_weight_clean)

다음 함수는 `Cdf`를 받아 `inverse` 메서드로 50백분위수를 구합니다. 이것이 중앙값입니다. 정확히는 데이터셋의 중앙값을 정의하는 방법 중 하나입니다.

In [54]:
def median(cdf):
    m = cdf.inverse(0.5)
    return m

두 자료의 중앙값을 구해 보겠습니다.

In [55]:
median(cdf_total_weight_bogus), median(cdf_total_weight_clean)

결과가 같습니다. 이 예에서는 이상치가 중앙값에 전혀 영향을 주지 않았습니다. 일반적으로 이상치는 평균보다 중앙값에 적은 영향을 줍니다.

**사분위범위**(interquartile range, IQR)는 75백분위수와 25백분위수의 차이입니다. 다음 함수는 `Cdf`를 받아 IQR을 반환합니다.

In [56]:
def iqr(cdf):
    low, high = cdf.inverse([0.25, 0.75])
    return high - low

두 자료의 사분위범위는 다음과 같습니다.

In [57]:
iqr(cdf_total_weight_bogus), iqr(cdf_total_weight_clean)

일반적으로 이상치는 표준편차보다 IQR에 적은 영향을 줍니다. 이 예에서는 영향이 전혀 없습니다.

마지막으로 다음 함수는 세 가지 통계량을 이용해 사분위 왜도를 구합니다.

- 중앙값
- 25백분위수와 75백분위수의 중간값
- IQR의 절반인 반사분위범위

In [58]:
def quartile_skewness(cdf):
    low, median, high = cdf.inverse([0.25, 0.5, 0.75])
    midpoint = (high + low) / 2
    semi_iqr = (high - low) / 2
    return (midpoint - median) / semi_iqr

두 자료의 사분위 왜도는 다음과 같습니다.

In [59]:
qskew1 = quartile_skewness(cdf_total_weight_bogus)
qskew2 = quartile_skewness(cdf_total_weight_clean)
qskew1, qskew2

이 예에서 소수의 이상치는 사분위 왜도에도 영향을 주지 않았습니다. 백분위수에 바탕을 둔 통계량이 이상치와 자료 오류에 덜 민감하다는 것을 보여줍니다.

## 난수

`Cdf` 객체로 분포를 따르는 난수를 효율적으로 생성할 수 있습니다. 먼저 0~1 사이의 균등분포에서 난수를 생성합니다. 그런 다음 그 값들을 역 CDF에 넣습니다. 다음 함수가 이 알고리즘을 구현합니다.

In [60]:
def sample_from_cdf(cdf, n):
    ps = np.random.random(size=n)
    return cdf.inverse(ps)

예로 달리기 속도의 무작위 표본을 만들어 보겠습니다.

In [61]:
sample = sample_from_cdf(cdf_speeds, 1001)

잘 되었는지 확인하려면 표본과 원래 자료의 CDF를 비교하면 됩니다.

In [62]:
cdf_sample = Cdf.from_seq(sample)

cdf_speeds.plot(label="original", ls="--")
cdf_sample.plot(label="sample", alpha=0.5)

decorate(xlabel="Speed (mph)", ylabel="CDF")

표본이 원래 자료의 분포를 따라갑니다. 알고리즘의 작동 방식을 이해하려면 다음 질문을 생각해 보세요. 달리기 속도 모집단에서 무작위 표본을 뽑고, 표본 속도들의 백분위 순위를 구했다고 하겠습니다. 이 백분위 순위들로 CDF를 만들면 어떤 모양일까요?

직접 확인해 보겠습니다. 앞서 생성한 표본의 백분위 순위는 다음과 같습니다.

In [63]:
percentile_ranks = cdf_speeds(sample) * 100

이 백분위 순위의 CDF를 그립니다.

In [64]:
cdf_percentile_rank = Cdf.from_seq(percentile_ranks)
cdf_percentile_rank.plot()

decorate(xlabel="Percentile rank", ylabel="CDF")

백분위 순위의 CDF는 0에서 1로 올라가는 직선에 가깝습니다. 어느 분포든 백분위 순위가 50%보다 낮은 값의 비율은 0.5이고, 90%보다 낮은 값의 비율은 0.9라고 생각하면 자연스럽습니다.

> 번역자 주: 동점이 없는 연속분포에서는 이 설명이 정확히 성립합니다. 여기처럼 값이 반복되는 경험분포에서는 누적확률이 뛰어오르므로, 백분위 순위가 정확한 균등분포가 되는 것은 아닙니다.

`Cdf`의 `sample` 메서드도 이 알고리즘을 사용하므로 다음과 같이 표본을 만들 수도 있습니다.

In [65]:
sample = cdf_speeds.sample(1001)

## 용어 정리

- **백분위 순위**(percentile rank): 분포에서 주어진 값 이하인 값의 비율을 백분율로 나타낸 것.
- **백분위수**(percentile): 주어진 백분위 순위에 대응하는 분포의 값.
- **누적분포함수**(cumulative distribution function, CDF): 어떤 값에 그 값 이하인 값들의 비율을 대응시키는 함수.
- **분위수**(quantile): 주어진 비율에 대응하는 분포의 값. 역 CDF로 구합니다.
- **강건성**(robust): 극단적인 값이나 이상치의 영향을 덜 받는 성질.
- **사분위범위**(interquartile range, IQR): 분포가 퍼진 정도를 나타내는 75백분위수와 25백분위수의 차이.

## 연습문제

### 연습문제 4.1

태어났을 때 몸무게가 얼마였나요? 모른다면 어머니나 아는 사람에게 물어보세요. 아무도 모르면 이 연습에서는 제 출생 체중인 8.5파운드를 써도 됩니다.

NSFG의 생존 출산 전체 자료로 출생 체중의 분포를 구하고, 본인의 백분위 순위를 찾으세요. 첫아이였다면 첫아이의 분포에서, 아니라면 나머지 아이의 분포에서 백분위 순위를 구하세요. 90백분위 이상이면 어머니께 다시 전화해 사과드리세요.

In [66]:
from nsfg import get_nsfg_groups

live, firsts, others = get_nsfg_groups()

In [67]:
# Solution goes here

In [68]:
# Solution goes here

In [69]:
# Solution goes here

### 연습문제 4.2

NSFG 자료의 생존 출산에서 `babysex` 열은 아기의 성별이 남성인지 여성인지 나타냅니다. `query`로 남아와 여아에 해당하는 행을 선택할 수 있습니다.

In [70]:
male = live.query("babysex == 1")
female = live.query("babysex == 2")
len(male), len(female)

남아와 여아의 출생 체중 분포를 나타내는 `Cdf` 객체를 만들고 두 CDF를 그리세요. 분포의 모양과 위치에는 어떤 차이가 있나요?

남아의 체중이 8.5파운드이면 백분위 순위는 얼마인가요? 같은 백분위 순위인 여아의 체중은 얼마인가요?

In [71]:
# Solution goes here

In [72]:
# Solution goes here

In [73]:
# Solution goes here

In [74]:
# Solution goes here

In [75]:
# Solution goes here

### 연습문제 4.3

NSFG 임신 자료에서 `agepreg` 열을 선택하고, 각 임신의 나이 분포를 나타내는 `Cdf`를 만드세요. CDF로 20세 이하의 비율과 30세 이하의 비율을 구하세요. 두 결과를 이용해 20세 초과 30세 이하인 비율을 구하세요.

> 번역자 주: 원문은 이 대목을 “임신 당시 나이”라고 표현하지만, `agepreg`는 1·2장에서 설명했듯 임신이 끝났을 때의 나이입니다.

In [76]:
from nsfg import read_fem_preg

preg = read_fem_preg()

In [77]:
# Solution goes here

In [78]:
# Solution goes here

In [79]:
# Solution goes here

In [80]:
# Solution goes here

### 연습문제 4.4

다음은 이 장 앞부분에서 소개한 James Joyce Ramble 완주자들의 달리기 속도입니다.

In [81]:
speeds = results["MPH"].values

이 속도 분포를 나타내는 `Cdf`를 만들고 중앙값, IQR, 사분위 왜도를 구하세요. 분포는 왼쪽으로 치우쳤나요, 오른쪽으로 치우쳤나요?

In [82]:
# Solution goes here

In [83]:
# Solution goes here

### 연습문제 4.5

`np.random.random`으로 생성한 수는 0~1 사이의 균등분포를 따라야 합니다. 그렇다면 표본의 CDF는 직선에 가까워야 합니다. 실제로 그런지 확인해 보겠습니다. 다음은 난수 1,001개로 된 표본입니다. 이 표본의 CDF를 그리세요. 직선처럼 보이나요?

In [84]:
t = np.random.random(1001)

In [85]:
# Solution goes here

In [86]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)